# ClimaLens AI: Recommender System Training

This notebook generates 50,000 synthetic trip queries using Python's `multiprocessing` and trains an XGBoost Regressor to predict the perfect `match_score`. You can use this directly in Google Colab!

In [ ]:
!pip install xgboost pandas numpy tqdm

In [ ]:
import pandas as pd
import numpy as np
import random
import multiprocessing
from tqdm.notebook import tqdm
import os

DESTINATION_KNOWLEDGE = {
    'Mirissa': {'interests': ['surfing', 'beaches', 'food'], 'best_months': [12, 1, 2, 3, 4], 'is_coastal': 1, 'is_hill': 0},
    'Weligama': {'interests': ['surfing', 'beaches'], 'best_months': [12, 1, 2, 3, 4], 'is_coastal': 1, 'is_hill': 0},
    'Unawatuna': {'interests': ['beaches', 'food'], 'best_months': [12, 1, 2, 3, 4], 'is_coastal': 1, 'is_hill': 0},
    'Galle Fort': {'interests': ['culture', 'food'], 'best_months': [12, 1, 2, 3, 4], 'is_coastal': 1, 'is_hill': 0},
    'Arugam Bay': {'interests': ['surfing', 'beaches'], 'best_months': [5, 6, 7, 8, 9], 'is_coastal': 1, 'is_hill': 0},
    'Trincomalee': {'interests': ['beaches', 'culture', 'wildlife'], 'best_months': [5, 6, 7, 8, 9], 'is_coastal': 1, 'is_hill': 0},
    'Yala': {'interests': ['wildlife'], 'best_months': [2, 3, 4, 5, 6, 7], 'is_coastal': 0, 'is_hill': 0},
    'Udawalawe': {'interests': ['wildlife'], 'best_months': list(range(1, 13)), 'is_coastal': 0, 'is_hill': 0},
    'Ella': {'interests': ['hiking', 'food'], 'best_months': [1, 2, 3, 7, 8], 'is_coastal': 0, 'is_hill': 1},
    'Nuwara Eliya': {'interests': ['culture', 'hiking'], 'best_months': [2, 3, 4], 'is_coastal': 0, 'is_hill': 1},
    'Sigiriya': {'interests': ['culture', 'hiking'], 'best_months': [1, 2, 3, 7, 8], 'is_coastal': 0, 'is_hill': 0},
    'Kandy': {'interests': ['culture', 'food'], 'best_months': [12, 1, 2, 3, 7, 8], 'is_coastal': 0, 'is_hill': 1}
}
ALL_INTERESTS = ['surfing', 'wildlife', 'hiking', 'culture', 'beaches', 'food']

def generate_batch(batch_size):
    rows = []
    dest_names = list(DESTINATION_KNOWLEDGE.keys())
    for _ in range(batch_size):
        month = random.randint(1, 12)
        user_interests = random.sample(ALL_INTERESTS, random.randint(1, 3))
        dest_name = random.choice(dest_names)
        dest_kb = DESTINATION_KNOWLEDGE[dest_name]
        
        score = 0
        for i in user_interests:
            if i in dest_kb['interests']:
                score += 35
        if month in dest_kb['best_months']:
            score += 25
        else:
            score -= 10
        score += random.uniform(-5, 5)
        score = max(0, min(100, score))
        
        rows.append({
            'user_month': month,
            'want_surfing': 1 if 'surfing' in user_interests else 0,
            'want_wildlife': 1 if 'wildlife' in user_interests else 0,
            'want_hiking': 1 if 'hiking' in user_interests else 0,
            'want_culture': 1 if 'culture' in user_interests else 0,
            'want_beaches': 1 if 'beaches' in user_interests else 0,
            'want_food': 1 if 'food' in user_interests else 0,
            'dest_is_coastal': dest_kb['is_coastal'],
            'dest_is_hill': dest_kb['is_hill'],
            'dest_has_surfing': 1 if 'surfing' in dest_kb['interests'] else 0,
            'dest_has_wildlife': 1 if 'wildlife' in dest_kb['interests'] else 0,
            'dest_has_hiking': 1 if 'hiking' in dest_kb['interests'] else 0,
            'dest_has_culture': 1 if 'culture' in dest_kb['interests'] else 0,
            'dest_has_beaches': 1 if 'beaches' in dest_kb['interests'] else 0,
            'dest_has_food': 1 if 'food' in dest_kb['interests'] else 0,
            'dest_is_best_month': 1 if month in dest_kb['best_months'] else 0,
            'target_score': round(score, 2)
        })
    return rows

TOTAL_SAMPLES = 50000
BATCH_SIZE = 5000
num_batches = TOTAL_SAMPLES // BATCH_SIZE
WORKERS = multiprocessing.cpu_count()
all_data = []
with multiprocessing.Pool(WORKERS) as pool:
    for result in tqdm(pool.imap_unordered(generate_batch, [BATCH_SIZE] * num_batches), total=num_batches, desc="Generating Trips"):
        all_data.extend(result)

df = pd.DataFrame(all_data)
df.head()

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X = df.drop('target_score', axis=1)
y = df['target_score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training XGBoost Recommender Model...")
recommender = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=200, max_depth=6, learning_rate=0.1)
recommender.fit(X_train, y_train)

preds = recommender.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"Recommender RMSE: {rmse:.2f}")

In [ ]:
import os
os.makedirs('models', exist_ok=True)
recommender.save_model('models/recommender_model.json')
print("\n\u2705 Model saved to models/recommender_model.json. You can download it now!")